# Fig. 1a | All-category catalyst-pathway Sankey diagram

This notebook redraws the single-panel all-category Sankey diagram using the latest
`database.xlsx` in the project root. Every original category is retained and no
low-frequency category is merged into Other.

In [ ]:
from pathlib import Path
import colorsys
import json

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.ticker import MaxNLocator
from scipy.stats import pearsonr, spearmanr
from matplotlib.colors import LinearSegmentedColormap
def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'database.xlsx').exists():
            return candidate
    raise FileNotFoundError('Could not locate database.xlsx in the current directory or its parents.')

ROOT = find_project_root()
OUT = ROOT / 'outputs' / 'figure1'
OUT.mkdir(parents=True, exist_ok=True)
DATA_PATH = ROOT / 'database.xlsx'
OUTPUT_DIR = OUT

# Four original main colors and restrained extensions.
COLOR_SYSTEM = {
    'red_dark': '#DB3124', 'red': '#E25A50', 'red_light': '#E87971', 'red_pale': '#ED9892',
    'blue_dark': '#4B74B2', 'blue': '#6F90C1', 'blue_light': '#8AA5CD', 'blue_pale': '#A5BAD8',
    'sky': '#90BEE0', 'sky_light': '#A6CBE6', 'sky_pale': '#C8DFF0',
    'yellow': '#FFDF92', 'yellow_light': '#FFE5A8', 'yellow_pale': '#FFEFC8',
    'gray_dark': '#666666', 'gray': '#999999', 'gray_light': '#D9D9D9', 'gray_pale': '#EEEEEE',
}

AM_COLORS = {
    'Ni': '#5E83BC',
    'Ru': '#DF665D',
    'Co': '#E1BD5F',
    'Pd': '#84A4CB',
    'Rh': '#D58F89',
    'Fe': '#A3BBD7',
    'Metal-free': '#AAAAAA',
}

DISPLAY_LABELS = {
    'Sp2-free': 'No secondary support', 'CB': 'CB',
    'Al2O3': r'Al$_2$O$_3$', 'CeO2': r'CeO$_2$', 'ZrO2': r'ZrO$_2$',
    'SiO2': r'SiO$_2$', 'TiO2': r'TiO$_2$', 'Cr2O3': r'Cr$_2$O$_3$',
    'Fe2O3': r'Fe$_2$O$_3$', 'Y2O3': r'Y$_2$O$_3$', 'Gd2O3': r'Gd$_2$O$_3$',
    'Mn3O4': r'Mn$_3$O$_4$','CB': 'CB',
}

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 10.0,
    'axes.titlesize': 12.0,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'savefig.facecolor': 'white',
})

assert DATA_PATH.exists(), f'Dataset not found: {DATA_PATH}'
print(f'Data: {DATA_PATH}')
print(f'Outputs: {OUTPUT_DIR}')

In [ ]:
data = pd.read_excel(DATA_PATH)
required = ['AM', 'Sp', 'Pr', 'PM']
missing = [col for col in required if col not in data.columns]
if missing:
    raise KeyError(f'Missing required columns: {missing}')

paths_raw = data[required].copy()
for col in required:
    paths_raw[col] = paths_raw[col].fillna('Missing').astype(str)

print(f'Rows used: {len(paths_raw):,}')
for col in required:
    print(f'{col}: {paths_raw[col].nunique()} categories')

In [ ]:
def prepare_paths(frame):
    """Retain every original category; no low-frequency class is merged into Other."""
    result = frame.copy()
    return result, {
        'supports': result['Sp'].value_counts().index.tolist(),
        'promoters': result['Pr'].value_counts().index.tolist(),
    }


main_paths, main_kept = prepare_paths(paths_raw)
zoom_source = paths_raw[(paths_raw['AM'] != 'Ni') | (paths_raw['Pr'] != 'Promoter-free')].copy()
zoom_paths, zoom_kept = prepare_paths(zoom_source)

print(f'Main panel rows: {len(main_paths):,}')
print(f'Zoom panel rows: {len(zoom_paths):,} ({len(zoom_paths)/len(paths_raw):.1%} of dataset)')
print(f'Main categories: AM={main_paths.AM.nunique()}, Sp={main_paths.Sp.nunique()}, Pr={main_paths.Pr.nunique()}, PM={main_paths.PM.nunique()}')
print(f'Zoom categories: AM={zoom_paths.AM.nunique()}, Sp={zoom_paths.Sp.nunique()}, Pr={zoom_paths.Pr.nunique()}, PM={zoom_paths.PM.nunique()}')


In [ ]:
def mix_with_white(color, amount=0.55):
    rgb = np.array(mpl.colors.to_rgb(color))
    return tuple(rgb * (1 - amount) + np.ones(3) * amount)


def ordered_categories(frame, column):
    return frame[column].value_counts().index.tolist()


def build_positions(frame, columns, gap=0.006):
    total = len(frame)
    positions, orders = {}, {}
    for column in columns:
        order = ordered_categories(frame, column)
        orders[column] = order
        usable = 1.0 - gap * max(len(order) - 1, 0)
        cursor = 1.0
        for category in order:
            height = usable * (frame[column] == category).sum() / total
            positions[(column, category)] = [cursor - height, cursor]
            cursor = cursor - height - gap
    return positions, orders


def spread_label_positions(targets, min_sep=0.033, lower=0.005, upper=0.995):
    """Distribute labels vertically while preserving their node order."""
    if not targets:
        return {}
    ordered = sorted(targets.items(), key=lambda item: item[1])
    categories = [item[0] for item in ordered]
    values = np.array([item[1] for item in ordered], dtype=float)
    if len(values) > 1:
        min_sep = min(min_sep, (upper - lower) / (len(values) - 1))
    values[0] = max(values[0], lower)
    for i in range(1, len(values)):
        values[i] = max(values[i], values[i - 1] + min_sep)
    if values[-1] > upper:
        values -= values[-1] - upper
        for i in range(len(values) - 2, -1, -1):
            values[i] = min(values[i], values[i + 1] - min_sep)
    return dict(zip(categories, values))


def flow_patch(ax, x0, x1, y0b, y0t, y1b, y1t, color, alpha=0.22):
    bend = (x1 - x0) * 0.46
    vertices = [
        (x0, y0b), (x0 + bend, y0b), (x1 - bend, y1b), (x1, y1b),
        (x1, y1t), (x1 - bend, y1t), (x0 + bend, y0t), (x0, y0t), (x0, y0b),
    ]
    codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
             MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CLOSEPOLY]
    ax.add_patch(PathPatch(MplPath(vertices, codes), facecolor=color, edgecolor='none', alpha=alpha, zorder=1))


def draw_alluvial(ax, frame, panel_label, gap=0.006, label_fontsize=8.2, label_min_sep=0.033):
    columns = ['AM', 'Sp', 'Pr', 'PM']
    x = dict(zip(columns, [0.0, 1.30, 2.60, 3.90]))
    positions, orders = build_positions(frame, columns, gap=gap)
    total, node_width = len(frame), 0.050

    for left, right in zip(columns[:-1], columns[1:]):
        pair_counts = frame.groupby([left, right], dropna=False).size().sort_values(ascending=False)
        left_offsets = {cat: positions[(left, cat)][0] for cat in orders[left]}
        right_offsets = {cat: positions[(right, cat)][0] for cat in orders[right]}
        usable_left = 1.0 - gap * max(len(orders[left]) - 1, 0)
        usable_right = 1.0 - gap * max(len(orders[right]) - 1, 0)
        for (source, target), count in pair_counts.items():
            h0, h1 = usable_left * count / total, usable_right * count / total
            y0b, y0t = left_offsets[source], left_offsets[source] + h0
            y1b, y1t = right_offsets[target], right_offsets[target] + h1
            source_rows = frame[(frame[left] == source) & (frame[right] == target)]
            am = source if left == 'AM' else source_rows['AM'].value_counts().index[0]
            flow_patch(ax, x[left] + node_width/2, x[right] - node_width/2, y0b, y0t, y1b, y1t,
                       AM_COLORS.get(am, COLOR_SYSTEM['gray']), alpha=0.25 if count/total < 0.03 else 0.40)
            left_offsets[source] += h0
            right_offsets[target] += h1

    for column in columns:
        for category in orders[column]:
            bottom, top = positions[(column, category)]
            if column == 'AM':
                face = mix_with_white(AM_COLORS.get(category, COLOR_SYSTEM['gray']), 0.12)
            elif category == 'Missing':
                face = COLOR_SYSTEM['gray_light']
            else:
                face = {'Sp': COLOR_SYSTEM['sky_light'], 'Pr': COLOR_SYSTEM['yellow'], 'PM': COLOR_SYSTEM['red_pale']}[column]
            ax.add_patch(Rectangle((x[column] - node_width/2, bottom), node_width, top-bottom,
                                   facecolor=face, edgecolor='white', linewidth=0.35, zorder=3))

    for column in columns:
        targets = {category: np.mean(positions[(column, category)]) for category in orders[column]}
        label_y = spread_label_positions(targets, min_sep=label_min_sep)
        left_side = column == 'AM'
        direction = -1 if left_side else 1
        node_edge = x[column] + direction * node_width / 2
        text_x = x[column] + direction * 0.075
        for category in orders[column]:
            center = targets[category]
            y = label_y[category]
            n = int((frame[column] == category).sum())
            share = n / total
            label = DISPLAY_LABELS.get(category, category)
            ax.plot([node_edge, text_x - direction * 0.010], [center, y],
                    color='#777777', linewidth=0.38, alpha=0.75, zorder=4, clip_on=False)
            ax.text(text_x, y, f'{label}, {share:.1%}',
            #ax.text(text_x, y, f'{label}  {n:,}, {share:.1%}',
                    ha='right' if left_side else 'left', va='center', fontsize=label_fontsize,
                    color='#222222', zorder=5, clip_on=False)

    headers = {'AM': 'Active metal', 'Sp': 'Support', 'Pr': 'Promoter', 'PM': 'Preparation method'}
    for column in columns:
        ax.text(x[column], 1.065, headers[column], ha='center', va='bottom', fontsize=20, fontweight='bold')
    ax.text(-0.70, 1.13, panel_label, ha='left', va='bottom', fontsize=12, fontweight='bold')
    ax.set_xlim(-0.72, 4.72)
    ax.set_ylim(-0.025, 1.18)
    ax.axis('off')


fig, ax_main = plt.subplots(figsize=(10.5, 10.5))

draw_alluvial(
    ax_main,
    main_paths,
    '',
    gap=0.006,
    label_fontsize=12,
    label_min_sep=0.03,
)

fig.subplots_adjust(left=0.025, right=0.985, top=0.95, bottom=0.035)
for suffix in ['svg', 'pdf', 'png']:
    kwargs = {'dpi': 600} if suffix == 'png' else {}
    fig.savefig(
        OUTPUT_DIR / f'Fig1a_all_categories_updated_database.{suffix}',
        bbox_inches='tight',
        pad_inches=0.08,
        **kwargs,
    )
plt.show()
print('Saved single-panel all-category SVG, PDF, and PNG outputs.')


In [ ]:
# Export the exact row-level pathway counts used by the figure.
main_table = (
    main_paths.groupby(['AM', 'Sp', 'Pr', 'PM'], dropna=False)
    .size()
    .reset_index(name='n')
    .sort_values('n', ascending=False)
)
main_table.to_csv(
    OUTPUT_DIR / 'Fig1a_all_categories_updated_full_dataset_paths.csv',
    index=False,
    encoding='utf-8-sig',
)

summary = {
    'dataset': str(DATA_PATH),
    'dataset_rows': int(len(paths_raw)),
    'all_categories_retained': True,
    'active_metals': main_paths['AM'].value_counts().to_dict(),
    'supports': main_paths['Sp'].value_counts().to_dict(),
    'promoters': main_paths['Pr'].value_counts().to_dict(),
    'preparation_methods': main_paths['PM'].value_counts().to_dict(),
    'display_only_label_changes': DISPLAY_LABELS,
}
(OUTPUT_DIR / 'Fig1a_all_categories_updated_run_summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding='utf-8',
)
print(main_table.head(12).to_string(index=False))


In [ ]:
COLOR_SYSTEM = {
    'red_dark': '#DB3124', 'red': '#E25A50', 'red_light': '#E87971', 'red_pale': '#ED9892',
    'blue_dark': '#4B74B2', 'blue': '#6F90C1', 'blue_light': '#8AA5CD', 'blue_pale': '#A5BAD8',
    'sky': '#90BEE0', 'sky_light': '#A6CBE6', 'sky_pale': '#C8DFF0',
    'yellow': '#FFDF92', 'yellow_light': '#FFE5A8', 'yellow_pale': '#FFEFC8',
    'gray_dark': '#666666', 'gray': '#999999', 'gray_light': '#D9D9D9', 'gray_pale': '#EEEEEE',
}

# Full names and units were checked against the manuscript feature table.
# MCP is the actual data-column name; the manuscript table uses MSP for the same feature.
LEFT_VARIABLES = [
    ('AMc', 'Active metal content (AMc)', 'wt.%'),
    ('Prc', 'Promoter content (Prc)', 'wt.%'),
    ('Spc', 'Primary support content (Spc)', 'wt.%'),
    ('Sp2c', 'Secondary support content (Sp2c)', 'wt.%'),
    ('MSP', 'Main support proportion (MSP)', '%'),
    ('CT', 'Calcination temperature (CT)', r'$^\circ$C'),
    ('Ct', 'Calcination time (Ct)', 'h'),
    ('RT', 'Reduction temperature (RT)', r'$^\circ$C'),
    ('RP', 'Reduction pressure (RP)', 'bar'),
    ('Rt', 'Reduction time (Rt)', 'h'),
    ('RH', r'Reduction H$_2$ concentration (RH)', '%'),
]
RIGHT_VARIABLES = [
    ('T', 'Reaction temperature (T)', r'$^\circ$C'),
    ('P', 'Reaction pressure (P)', 'bar'),
    ('GHSV', 'Gas hourly space velocity (GHSV)', r'L g$_{cat}^{-1}$ h$^{-1}$'),
    ('Inert', 'Inert gas in feed (Inert)', '%'),
    ('CO', 'CO in feed (CO)', '%'),
    ('CH4', r'CH$_4$ in feed (CH$_4$)', '%'),
    ('H2O', r'H$_2$O in feed (H$_2$O)', '%'),
    ('H2/CO2', r'H$_2$/CO$_2$ feed ratio (H$_2$/CO$_2$)', ''),
    ('Rate', r'CO$_2$ conversion rate (Rate)', '%'),
]
ALL_VARIABLES = LEFT_VARIABLES + RIGHT_VARIABLES

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 8.0,
    'axes.linewidth': 0.7,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'savefig.facecolor': 'white',
})

assert DATA_PATH.exists(), f'Dataset not found: {DATA_PATH}'
print(f'Data: {DATA_PATH}')

In [ ]:
columns = [item[0] for item in ALL_VARIABLES]
missing = [column for column in columns if column not in data.columns]
if missing:
    raise KeyError(f'Missing required columns: {missing}')
numeric = data[columns].apply(pd.to_numeric, errors='coerce')
rows = []
for column, label, unit in ALL_VARIABLES:
    values = numeric[column].dropna()
    q = values.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    rows.append({
        'variable': column, 'label': label, 'unit': unit, 'n': len(values),
        'missing_rate': numeric[column].isna().mean(),
        'zero_rate': (values == 0).mean(),
        'min': values.min(), 'q01': q.loc[0.01], 'q05': q.loc[0.05],
        'q25': q.loc[0.25], 'median': q.loc[0.50], 'q75': q.loc[0.75],
        'q95': q.loc[0.95], 'q99': q.loc[0.99], 'max': values.max(),
    })
summary = pd.DataFrame(rows).set_index('variable')
summary

In [ ]:
def draw_variable_axis(ax, row, label, unit, color):
    # Full observed range remains visible, which is important for sparse feed variables.
    ax.hlines(0, row['min'], row['max'], color=COLOR_SYSTEM['gray_light'], linewidth=1.15, zorder=1)
    ax.hlines(0, row.q05, row.q95, color=color, linewidth=1.8, alpha=0.68, zorder=2)
    ax.hlines(0, row.q25, row.q75, color=color, linewidth=7.0, alpha=0.88, zorder=3)
    ax.scatter(row['median'], 0, s=39, color=color, edgecolor='white', linewidth=0.75, zorder=4)
    ax.plot([row.q01, row.q99], [0, 0], '|', color=COLOR_SYSTEM['gray_dark'], markersize=6, markeredgewidth=0.75)

    ax.set_yticks([])
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.tick_params(axis='x', labelsize=10.0, length=2.5, pad=2)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_color('#888888')
    ax.spines['bottom'].set_linewidth(0.60)

    unit_text = f' [{unit}]' if unit else ''
    ax.text(-0.01, 0.62, label + unit_text, transform=ax.transAxes,
            ha='left', va='bottom', fontsize=12)
    notes = []
    if row.missing_rate >= 0.005:
        notes.append(f'Missing {row.missing_rate:.1%}')
    if row.zero_rate >= 0.10:
        notes.append(f'Zero {row.zero_rate:.1%}')
    if notes:
        ax.text(1.0, 0.62, ' | '.join(notes), transform=ax.transAxes,
                ha='right', va='bottom', fontsize=8, color=COLOR_SYSTEM['gray_dark'])


fig = plt.figure(figsize=(10, 10.0))
outer = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.0], wspace=0.18)
left_grid = outer[0].subgridspec(len(LEFT_VARIABLES), 1, hspace=0.82)
right_grid = outer[1].subgridspec(len(RIGHT_VARIABLES), 1, hspace=0.82)

#left_colors = [COLOR_SYSTEM['blue'], COLOR_SYSTEM['blue_light'], COLOR_SYSTEM['sky'],
#               COLOR_SYSTEM['sky_light'], COLOR_SYSTEM['blue_dark']] + [COLOR_SYSTEM['yellow']] * 6
#right_colors = [COLOR_SYSTEM['red'], COLOR_SYSTEM['red_light'], COLOR_SYSTEM['sky'],
#                COLOR_SYSTEM['red_pale'], COLOR_SYSTEM['blue_light'], COLOR_SYSTEM['blue_light'],
#                COLOR_SYSTEM['blue_light'], COLOR_SYSTEM['yellow'], COLOR_SYSTEM['red_dark']]
##
def color_gradient(start_color, end_color, n):
    """Generate n colors from light to dark."""
    start = np.array(mpl.colors.to_rgb(start_color))
    end = np.array(mpl.colors.to_rgb(end_color))
    return [
        mpl.colors.to_hex(start + (end - start) * i / max(n - 1, 1))
        for i in range(n)
    ]
# Five catalyst-composition variables: yellow gradient
composition_colors = color_gradient(start_color='#F8DE91',end_color='#D7A62A',n=5,)
# Six catalyst-preparation variables: blue gradient
preparation_colors = color_gradient(start_color='#B7D1E8',end_color='#4B74B2',n=6,)
# Eight reaction-operation variables: red gradient
operation_colors = color_gradient(start_color='#F3B5AE',end_color='#D7473E',n=8,)
# Final CO2 conversion rate: blue
conversion_color = '#D7A62A'
left_colors = composition_colors + preparation_colors
right_colors = operation_colors + [conversion_color]

##
left_axes = []
for index, ((column, label, unit), color) in enumerate(zip(LEFT_VARIABLES, left_colors)):
    ax = fig.add_subplot(left_grid[index, 0])
    draw_variable_axis(ax, summary.loc[column], label, unit, color)
    left_axes.append(ax)

right_axes = []
for index, ((column, label, unit), color) in enumerate(zip(RIGHT_VARIABLES, right_colors)):
    ax = fig.add_subplot(right_grid[index, 0])
    draw_variable_axis(ax, summary.loc[column], label, unit, color)
    right_axes.append(ax)

#left_axes[0].text(-0.07, 1.75, 'h', transform=left_axes[0].transAxes, fontsize=10.0, fontweight='bold')
#right_axes[0].text(-0.07, 1.75, 'i', transform=right_axes[0].transAxes, fontsize=10.0, fontweight='bold')

fig.subplots_adjust(left=0.055, right=0.985, top=0.955, bottom=0.055)
for suffix in ['svg', 'pdf', 'png']:
    kwargs = {'dpi': 600} if suffix == 'png' else {}
    fig.savefig(OUTPUT_DIR / f'Fig1_all_numeric_distributions.{suffix}', bbox_inches='tight', pad_inches=0.05, **kwargs)
plt.show()

In [ ]:
summary.reset_index().to_csv(OUTPUT_DIR / 'Fig1_all_numeric_distribution_summary.csv', index=False, encoding='utf-8-sig')
run_summary = {
    'dataset': str(DATA_PATH),
    'dataset_rows': int(len(data)),
    'variables': columns,
    'faint_line': 'observed minimum-maximum',
    'thin_colored_line': '5th-95th percentile',
    'thick_line': '25th-75th percentile',
    'circle': 'median',
    'end_ticks': '1st and 99th percentiles',
    'M': 'missing-value rate, shown when >= 0.5%',
    'Z': 'zero-value rate, shown when >= 10%',
    'naming_note': 'Data column MCP corresponds to manuscript-table MSP; figure uses MCP to match the dataset.',
}
(OUTPUT_DIR / 'Fig1_all_numeric_run_summary.json').write_text(json.dumps(run_summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(summary[['n', 'missing_rate', 'zero_rate', 'min', 'q05', 'median', 'q95', 'max']].to_string())

In [ ]:
COLOR_SYSTEM = {
    'red_dark': '#DB3124', 'red': '#E25A50', 'red_light': '#E87971', 'red_pale': '#ED9892',
    'blue_dark': '#4B74B2', 'blue': '#6F90C1', 'blue_light': '#8AA5CD', 'blue_pale': '#A5BAD8',
    'sky': '#90BEE0', 'sky_light': '#A6CBE6', 'sky_pale': '#C8DFF0',
    'yellow': '#FFDF92', 'yellow_light': '#FFE5A8', 'yellow_pale': '#FFEFC8',
    'gray_dark': '#666666', 'gray': '#999999', 'gray_light': '#D9D9D9', 'gray_pale': '#EEEEEE',
}

GROUPS = {
    'Composition': ['AMc', 'Prc', 'Spc', 'Sp2c', 'MSP'],
    'Preparation': ['CT', 'Ct', 'RT', 'RP', 'Rt', 'RH'],
    'Operating conditions': ['T', 'P', 'GHSV', 'Inert', 'CO', 'CH4', 'H2O', 'H2/CO2'],
}
FEATURES = [feature for group in GROUPS.values() for feature in group]

SHORT_LABELS = {
    'AMc': 'AMc', 'Prc': 'Prc', 'Spc': 'Spc', 'Sp2c': 'Sp2c', 'MSP': 'MSP',
    'CT': 'CT', 'Ct': 'Ct', 'RT': 'RT', 'RP': 'RP', 'Rt': 'Rt', 'RH': 'RH',
    'T': 'T', 'P': 'P', 'GHSV': 'GHSV', 'Inert': 'Inert', 'CO': 'CO',
    'CH4': 'CH4', 'H2O': 'H2O', 'H2/CO2': 'H2/CO2',
}

LONG_LABELS = {
    'AMc': 'Active-metal content', 'Prc': 'Promoter content',
    'Spc': 'Primary-support content', 'Sp2c': 'Secondary-support content',
    'MSP': 'Main-support proportion',
    'CT': 'Calcination temperature', 'Ct': 'Calcination time',
    'RT': 'Reduction temperature', 'RP': 'Reduction pressure',
    'Rt': 'Reduction time', 'RH': 'Reduction atmosphere',
    'T': 'Reaction temperature', 'P': 'Reaction pressure', 'GHSV': 'GHSV',
    'Inert': 'Inert-gas fraction', 'CO': 'CO fraction',
    'CH4': 'CH4 fraction', 'H2O': 'H2O fraction',
    'H2/CO2': 'H2/CO2 ratio',
}

GROUP_COLORS = {
    'Composition': COLOR_SYSTEM['blue_pale'],
    'Preparation': COLOR_SYSTEM['yellow_pale'],
    'Operating conditions': COLOR_SYSTEM['red_pale'],
}

In [ ]:
analysis_data = data[FEATURES + [TARGET]].apply(pd.to_numeric, errors='coerce').copy()

print(f'Dataset rows: {len(analysis_data):,}')
print(f'Features excluding prediction target: {len(FEATURES)}')
print('Missingness (%):')
print((analysis_data.isna().mean() * 100).round(2).to_string())

In [ ]:
def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full_like(p, np.nan)
    valid = np.isfinite(p)
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    correction = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    correction = np.minimum.accumulate(correction[::-1])[::-1]
    restored = np.empty_like(correction)
    restored[order] = np.minimum(correction, 1.0)
    adjusted[valid] = restored
    return adjusted

rows = []
all_columns = FEATURES + [TARGET]
for i, left in enumerate(all_columns):
    for right in all_columns[i + 1:]:
        pair = analysis_data[[left, right]].dropna()
        if len(pair) < 3 or pair[left].nunique() < 2 or pair[right].nunique() < 2:
            sr = sp = pr = pp = np.nan
        else:
            sr, sp = spearmanr(pair[left], pair[right])
            pr, pp = pearsonr(pair[left], pair[right])
        rows.append({'left': left, 'right': right, 'n': len(pair),
                     'spearman_rho': sr, 'spearman_p': sp,
                     'pearson_r': pr, 'pearson_p': pp})

statistics = pd.DataFrame(rows)
statistics['spearman_p_BH'] = benjamini_hochberg(statistics['spearman_p'])
statistics['pearson_p_BH'] = benjamini_hochberg(statistics['pearson_p'])
spearman_matrix = analysis_data[FEATURES].corr(method='spearman')

target_rows = []
for feature in FEATURES:
    row = statistics[((statistics['left'] == feature) & (statistics['right'] == TARGET)) |
                     ((statistics['right'] == feature) & (statistics['left'] == TARGET))].iloc[0]
    target_rows.append({'feature': feature, 'rho': row['spearman_rho'], 'n': int(row['n']),
                        'p': row['spearman_p'], 'p_BH': row['spearman_p_BH']})
target_corr = pd.DataFrame(target_rows).sort_values('rho')
display(target_corr)

In [ ]:
correlation_cmap = LinearSegmentedColormap.from_list(
    'soft_blue_white_red', [COLOR_SYSTEM['blue_dark'], '#F7F7F7', COLOR_SYSTEM['red_dark']], N=256
)

fig = plt.figure(figsize=(19.0, 12.0))
gs = fig.add_gridspec(1, 2, width_ratios=[1.48, 0.92], wspace=0.34)
ax_hm = fig.add_subplot(gs[0, 0])
ax_bar = fig.add_subplot(gs[0, 1])

# 热力图绘制
matrix = spearman_matrix.to_numpy()
mask = np.triu(np.ones_like(matrix, dtype=bool), k=0)
masked = np.ma.array(matrix, mask=mask)
im = ax_hm.imshow(masked, cmap=correlation_cmap, vmin=-1.0, vmax=1.0, interpolation='none')

for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        if i > j:
            ax_hm.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                      edgecolor='white', linewidth=0.55))
            value = matrix[i, j]
            if abs(value) >= 0.20:
                text_color = 'white' if abs(value) >= 0.55 else '#222222'
                ax_hm.text(j, i, f'{value:.2f}', ha='center', va='center',
                           fontsize=8.5, color=text_color)

labels = [SHORT_LABELS[x] for x in FEATURES]
ax_hm.set_xticks(range(len(FEATURES)), labels, rotation=48, ha='right', rotation_mode='anchor')
ax_hm.set_yticks(range(len(FEATURES)), labels)
ax_hm.tick_params(length=0, pad=4, labelsize=16)
for spine in ax_hm.spines.values():
    spine.set_visible(False)

start = 0
for name, members in GROUPS.items():
    end = start + len(members)
    ax_hm.plot([start - 0.45, end - 0.55], [-0.92, -0.92], color=GROUP_COLORS[name],
               linewidth=5.0, solid_capstyle='butt', clip_on=False)
    if end < len(FEATURES):
        ax_hm.axhline(end - 0.5, color='#B8B8B8', linewidth=0.65)
        ax_hm.axvline(end - 0.5, color='#B8B8B8', linewidth=0.65)
    start = end

cbar = fig.colorbar(im, ax=ax_hm, fraction=0.052, pad=0.032, shrink=0.90, aspect=24)
cbar.set_ticks([-1.0, -0.5, 0, 0.5, 1.0])
cbar.ax.tick_params(labelsize=18, length=3.5)
cbar.outline.set_linewidth(0.55)
ax_hm.text(-0.14, 1.045, 'c', transform=ax_hm.transAxes, fontsize=18.0, fontweight='bold')

# 柱状图绘制
plot = target_corr.reset_index(drop=True)
y = np.arange(len(plot))
colors = [COLOR_SYSTEM['red'] if value >= 0 else COLOR_SYSTEM['blue'] for value in plot['rho']]
ax_bar.axvline(0, color='#777777', linewidth=0.7, zorder=0)
for yi, value, color in zip(y, plot['rho'], colors):
    ax_bar.hlines(yi, 0, value, color=color, linewidth=1.8, alpha=0.80)
ax_bar.scatter(plot['rho'], y, s=48, color=colors, edgecolor='white', linewidth=0.55, zorder=3)
for yi, value in zip(y, plot['rho']):
    offset = 0.018 if value >= 0 else -0.018
    ax_bar.text(value + offset, yi, f'{value:.2f}', va='center',
                ha='left' if value >= 0 else 'right', fontsize=15)
ax_bar.set_yticks(y, [SHORT_LABELS[x] for x in plot['feature']])
ax_bar.set_xlim(-0.40, 0.64)
ax_bar.set_xlabel(r'Spearman $\rho$ with CO$_2$ conversion rate', fontsize=22)
ax_bar.tick_params(axis='y', length=0, pad=5, labelsize=16)
ax_bar.tick_params(axis='x', labelsize=20)
ax_bar.grid(axis='x', color='#E6E6E6', linewidth=0.5)
ax_bar.set_axisbelow(True)
ax_bar.spines['left'].set_visible(False)
ax_bar.spines['bottom'].set_linewidth(0.7)
ax_bar.text(-0.18, 1.045, 'd', transform=ax_bar.transAxes, fontsize=10.0, fontweight='bold')

fig.subplots_adjust(left=0.070, right=0.985, top=0.94, bottom=0.15)

# 统一保存到全局定义的 OUTPUT_DIR
for suffix in ['svg', 'pdf', 'png']:
    kwargs = {'dpi': 600} if suffix == 'png' else {}
    fig.savefig(OUTPUT_DIR / f'Fig1cd_all_factors_MSP_spearman.{suffix}',
                bbox_inches='tight', pad_inches=0.06, **kwargs)
plt.show()

# 导出数据文件
statistics.to_csv(OUTPUT_DIR / 'Fig1cd_all_factors_pairwise_correlations.csv', index=False, encoding='utf-8-sig')
spearman_matrix.to_csv(OUTPUT_DIR / 'Fig1c_all_factors_spearman_matrix.csv', encoding='utf-8-sig')
target_corr.to_csv(OUTPUT_DIR / 'Fig1d_all_factors_target_ranking.csv', index=False, encoding='utf-8-sig')

In [ ]:
statistics.to_csv(OUTPUT_DIR / 'Fig1cd_all_factors_pairwise_correlations.csv',
                  index=False, encoding='utf-8-sig')
spearman_matrix.to_csv(OUTPUT_DIR / 'Fig1c_all_factors_spearman_matrix.csv', encoding='utf-8-sig')
target_corr.to_csv(OUTPUT_DIR / 'Fig1d_all_factors_target_ranking.csv',
                   index=False, encoding='utf-8-sig')

summary = {
    'dataset': str(DATA_PATH),
    'dataset_rows': int(len(analysis_data)),
    'features_excluding_prediction_target': FEATURES,
    'feature_count': len(FEATURES),
    'prediction_target': TARGET,
    'database_column': 'MSP',
    'primary_method': 'Spearman pairwise-complete correlation',
    'matrix_annotation_threshold': 'absolute rho >= 0.20',
    'multiple_testing_correction_in_source_table': 'Benjamini-Hochberg',
}
(OUTPUT_DIR / 'Fig1cd_all_factors_MSP_run_summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
print(target_corr.sort_values('rho', ascending=False).to_string(index=False))